In [3]:
import os
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
from pinecone import Pinecone, ServerlessSpec
from pypdf import PdfReader

load_dotenv()
API_KEY = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")  # Ensure this is in your .env

# Initialize clients
client = OpenAI(api_key=API_KEY)
pc = Pinecone(api_key=PINECONE_API_KEY)


In [4]:
# Define Index Configurations
INDEX_NAME = "pdf-rag-index"
DIMENSION = 1536  # Dimension for 'text-embedding-3-small'


# -------- READ PDF --------
def read_pdf(file):
    reader = PdfReader(file)
    text = ""
    for page in reader.pages:
        text += page.extract_text() or ""
    return text


# -------- CHUNK TEXT --------
def chunk_text(text, size=300):
    return [text[i : i + size] for i in range(0, len(text), size)]


# -------- EMBEDDINGS --------
def embed(texts):
    response = client.embeddings.create(model="text-embedding-3-small", input=texts)
    return np.array([d.embedding for d in response.data]).astype("float32")


# -------- BUILD VECTOR DB (PINECONE) --------
def build_index(chunks):
    # 1. Create index if it doesn't exist
    if INDEX_NAME not in pc.list_indexes().names():
        pc.create_index(
            name=INDEX_NAME,
            dimension=DIMENSION,
            metric="cosine",  # Cosine is recommended for OpenAI embeddings
            spec=ServerlessSpec(cloud="aws", region="us-east-1"),
        )

    index = pc.Index(INDEX_NAME)

    # 2. Generate embeddings
    embeddings = embed(chunks)

    # 3. Format data for Pinecone upsert: (id, vector, metadata)
    vectors_to_upsert = []
    for i, (chunk, embedding) in enumerate(zip(chunks, embeddings)):
        vectors_to_upsert.append(
            {"id": f"chunk-{i}", "values": embedding.tolist(), "metadata": {"text": chunk}}
        )

    # 4. Upload to cloud
    index.upsert(vectors = vectors_to_upsert)
    return index


# -------- RETRIEVE --------
def retrieve(query, index, k=3):
    q_emb = embed([query])[0].tolist()

    # Query Pinecone and request metadata to get the original text back
    results = index.query(vector=q_emb, top_k=k, include_metadata=True)

    return [match["metadata"]["text"] for match in results["matches"]]


# -------- ASK LLM --------
def ask_llm(question, context):
    prompt = f"""
    Answer using the context below. Keep it short.

    Context:
    {context}

    Question: {question}
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini", messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content


# TEST IMPLEMENTATION 

In [5]:

pdf_path = "document-loxford-company.pdf"

print("Step 1: Extracting text from document...")
raw_text = read_pdf(pdf_path)

raw_text

Step 1: Extracting text from document...


'Loxford Technologies – Company Overview \nLoxford Technologies is a global technology solutions provider specializing in \nartificial intelligence, data analytics, and enterprise automation. Founded in 2012, \nthe company has grown rapidly into a mid-sized enterprise serving clients across \nNorth America, Europe, and Asia. \nLoxford Technologies was founded by Daniel Reeves, a former data scientist, and \nAnika Lomax, a software engineer with a background in distributed systems. The \nfounders envisioned a company that could bridge the gap between cutting-edge \nAI research and real-world business applications. \n• CEO: Daniel Reeves \n• CTO: Anika Lomax \nThe company is headquartered in Austin, Texas, USA, with additional offices in: \n• London, UK \n• Bengaluru, India \n• Berlin, Germany \nAs of 2024, Loxford Technologies reported an estimated annual revenue of $180 \nmillion, with a year-over-year growth rate of approximately 22%. The company \nemploys over 850 professionals, incl

In [8]:
print("Step 2: Splitting text into chunks...")
chunks = chunk_text(raw_text, size=300)
print(f"Generated {len(chunks)} text chunks.")
print("+++++++++++++++++++++++")

for i,c in enumerate(chunks):
    print(f"Chunk {i+1}:\n", c)
    print("-------------------\n")


Step 2: Splitting text into chunks...
Generated 10 text chunks.
+++++++++++++++++++++++
Chunk 1:
 Loxford Technologies – Company Overview 
Loxford Technologies is a global technology solutions provider specializing in 
artificial intelligence, data analytics, and enterprise automation. Founded in 2012, 
the company has grown rapidly into a mid-sized enterprise serving clients across 
North Ameri
-------------------

Chunk 2:
 ca, Europe, and Asia. 
Loxford Technologies was founded by Daniel Reeves, a former data scientist, and 
Anika Lomax, a software engineer with a background in distributed systems. The 
founders envisioned a company that could bridge the gap between cutting-edge 
AI research and real-world business ap
-------------------

Chunk 3:
 plications. 
• CEO: Daniel Reeves 
• CTO: Anika Lomax 
The company is headquartered in Austin, Texas, USA, with additional offices in: 
• London, UK 
• Bengaluru, India 
• Berlin, Germany 
As of 2024, Loxford Technologies reported an esti

In [9]:
print("Step 3: Initializing and uploading embeddings to Pinecone...")
# This will create the index online if it's your first time running it
pinecone_index = build_index(chunks)
print("Upload complete!")


Step 3: Initializing and uploading embeddings to Pinecone...
Upload complete!


In [15]:
# 4. Define your testing query
query = "Who was the founder Loxford Tech company?"
print(f"\nStep 4: Querying the database for: '{query}'")

# Fetch context from Pinecone
retrieved_contexts = retrieve(query, pinecone_index, k=2)
combined_context = "\n".join(retrieved_contexts)



Step 4: Querying the database for: 'Who was the founder Loxford Tech company?'


In [16]:
retrieved_contexts

['Loxford Technologies – Company Overview \nLoxford Technologies is a global technology solutions provider specializing in \nartificial intelligence, data analytics, and enterprise automation. Founded in 2012, \nthe company has grown rapidly into a mid-sized enterprise serving clients across \nNorth Ameri',
 'plications. \n• CEO: Daniel Reeves \n• CTO: Anika Lomax \nThe company is headquartered in Austin, Texas, USA, with additional offices in: \n• London, UK \n• Bengaluru, India \n• Berlin, Germany \nAs of 2024, Loxford Technologies reported an estimated annual revenue of $180 \nmillion, with a year-over-year g']

In [17]:
combined_context

'Loxford Technologies – Company Overview \nLoxford Technologies is a global technology solutions provider specializing in \nartificial intelligence, data analytics, and enterprise automation. Founded in 2012, \nthe company has grown rapidly into a mid-sized enterprise serving clients across \nNorth Ameri\nplications. \n• CEO: Daniel Reeves \n• CTO: Anika Lomax \nThe company is headquartered in Austin, Texas, USA, with additional offices in: \n• London, UK \n• Bengaluru, India \n• Berlin, Germany \nAs of 2024, Loxford Technologies reported an estimated annual revenue of $180 \nmillion, with a year-over-year g'

In [18]:

print("\n--- Retrieved Context Matches ---")
for idx, match in enumerate(retrieved_contexts):
    print(f"[{idx+1}] {match}")



--- Retrieved Context Matches ---
[1] Loxford Technologies – Company Overview 
Loxford Technologies is a global technology solutions provider specializing in 
artificial intelligence, data analytics, and enterprise automation. Founded in 2012, 
the company has grown rapidly into a mid-sized enterprise serving clients across 
North Ameri
[2] plications. 
• CEO: Daniel Reeves 
• CTO: Anika Lomax 
The company is headquartered in Austin, Texas, USA, with additional offices in: 
• London, UK 
• Bengaluru, India 
• Berlin, Germany 
As of 2024, Loxford Technologies reported an estimated annual revenue of $180 
million, with a year-over-year g


In [19]:

print("\nStep 5: Sending context to GPT-4o-mini...")
answer = ask_llm(query, combined_context)

print("\n--- Final Answer from LLM ---")
print(answer)



Step 5: Sending context to GPT-4o-mini...

--- Final Answer from LLM ---
Loxford Technologies was founded by Daniel Reeves.


# Delete the index

In [20]:
# Check and delete the index
if INDEX_NAME in pc.list_indexes().names():
    print(f"Deleting index '{INDEX_NAME}'...")
    pc.delete_index(INDEX_NAME)
    print("Index deleted successfully!")
else:
    print(f"Index '{INDEX_NAME}' does not exist.")

Deleting index 'pdf-rag-index'...
Index deleted successfully!
